In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy

from scipy.stats import chi2_contingency
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import  TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from scipy.sparse import hstack
from imblearn.over_sampling import SMOTE
from sklearn.metrics import confusion_matrix

import re
import nltk
import logging
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import PorterStemmer
nltk.download('vader_lexicon')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ghimi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ghimi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ghimi\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import string
from nltk.sentiment import SentimentIntensityAnalyzer
sia=SentimentIntensityAnalyzer()
logging.basicConfig(level=logging.INFO)
logger=logging.getLogger(__name__)

In [ ]:
# loading the dataset and checking unique terms in 'status' column of dataset
# checking null values
file_path = "D:\\Combined Data.csv"
df = pd.read_csv(file_path)
print(df['status'].unique())

print(df.isnull().sum())

<StringArray>
[             'Anxiety',               'Normal',           'Depression',
             'Suicidal',               'Stress',              'Bipolar',
 'Personality disorder']
Length: 7, dtype: str
Unnamed: 0      0
statement     362
status          0
dtype: int64


In [ ]:
# Mapping the stress status to corresponding level loww,high,medium,extreme
stress_map={
    'Normal':'low',
    'Anxiety':'Medium',
    'Stress':'Medium',
    'Depression':'High',
    'Suicidal':'Extreme',
    'Bipolar':'Extreme',
    'Personality disorder':'Extreme',
}

df['stress_level']=df['status'].map(stress_map)

In [ ]:
df=df.dropna(subset=['stress_level'])
print(df['stress_level'].value_counts())

In [ ]:


#Preprocessing task
# view first rows
print(df.head())

# initialize stemmer
stemmer = PorterStemmer()

# stopwords
stop_words = set(stopwords.words('english'))

# preprocessing function
def preprocess_text_tfidf(text):

    # convert to string
    text = str(text)

    # lowercase
    text = text.lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # remove numbers
    text = re.sub(r'\d+', '', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokenize
    words = text.split()

    # remove stopwords and apply stemming
    words=[stemmer.stem(w) for w in words if w not in stop_words]
  
    # join back into sentence
    text = " ".join(words)

    return text

def preprocess_text_vader(text):
    text=str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    return text

df['clean_text']=df['statement'].apply(preprocess_text_tfidf)
df['vader_text']=df['statement'].apply(preprocess_text_vader)
df=df[df['clean_text'].str.strip().str.len()>0]

print("\nSample preprocessed texts:")
print(df[['statement', 'clean_text','vader_text']].head())
                    




In [ ]:
# printing the number/ count of different stress level
print(df['stress_level'].value_counts())


NameError: name 'df' is not defined

In [ ]:
# printing the head of the dataframe
print(df.head())

In [ ]:
# designing functions to  get sentiment scores for text
def get_vader_scores(text):
    scores=sia.polarity_scores(text)
    return{
        'vader_compound':scores['compound'],
        'vader_neg':scores['neg'],
        'vader_pos': scores['pos'],
        'vader_neu': scores['neu']

    }
vader_scores_list=df['vader_text'].apply(get_vader_scores)
vader_df=pd.DataFrame(vader_scores_list.tolist())

print("\nVader scores samples")
print(vader_df.head())




IndentationError: unexpected indent (235701357.py, line 2)

In [ ]:
X_text=df['clean_text'].reset_index(drop=True)
y=df['stress_level'].reset_index(drop=True)
vader_features=vader_df.reset_index(drop=True)
# splitting data for testing

X_train_text, X_test_text, y_train, y_test, vader_train, vader_test = train_test_split(
    X_text, y, vader_features,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [ ]:
# tfidf vectorizer that converts string to numerical values
vectorizer=TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,3),
    stop_words=None,
    sublinear_tf=True,
    min_df=2
    )

X_train_tfidf=vectorizer.fit_transform(X_train_text)
X_test_tfidf=vectorizer.transform(X_test_text)




In [ ]:
# combining the train values of tfidf and vader
X_train=hstack([X_train_tfidf,vader_train.values])
X_test=hstack([X_test_tfidf,vader_test.values])
logger.info(f"Combined feature shape-Train:{X_train.shape},Test:{X_test.shape}")



INFO:__main__:Combined feature shape-Train:(42311, 10004),Test:(10578, 10004)


In [17]:
#balancing training set with smote
try:
    smote=SMOTE(random_state=42,k_neighbors=5)
    X_train_balanced,y_train_balanced=smote.fit_resample(X_train,y_train)
    logger.info(f"After SMOTE - Train distribution:\n{pd.Series(y_train_balanced).value_counts()}")
    print(f"\nAfter SMOTE resampling:")
    print(pd.Series(y_train_balanced).value_counts())
except Exception as e:
    logger.warning(f"SMOTE failed: {e}. Using original training data.")
    X_train_balanced = X_train
    y_train_balanced = y_train




INFO:__main__:After SMOTE - Train distribution:
stress_level
High       12971
low        12971
Medium     12971
Extreme    12971
Name: count, dtype: int64



After SMOTE resampling:
stress_level
High       12971
low        12971
Medium     12971
Extreme    12971
Name: count, dtype: int64


In [ ]:
#Random forest classifeir for stress classification
model=RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    max_depth=70,
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_balanced,y_train_balanced)
y_pred = model.predict(X_test)
 
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
logger.info(f"Test Accuracy: {accuracy:.4f}")

INFO:__main__:Test Accuracy: 0.7533


In [ ]:

# printing classification report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

     Extreme       0.71      0.56      0.63      2945
        High       0.71      0.73      0.72      3079
      Medium       0.72      0.79      0.76      1311
         low       0.83      0.93      0.88      3243

    accuracy                           0.75     10578
   macro avg       0.74      0.75      0.75     10578
weighted avg       0.75      0.75      0.75     10578



In [ ]:
# confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=["low", "Medium", "High", "Extreme"])
cm_df = pd.DataFrame(cm,
    index=["Actual: Low", "Actual: Medium", "Actual: High", "Actual: Extreme"],
    columns=["Pred: Low", "Pred: Medium", "Pred: High", "Pred: Extreme"]
)
print(cm_df)


Confusion Matrix:
                 Pred: Low  Pred: Medium  Pred: High  Pred: Extreme
Actual: Low           3023            83          72             65
Actual: Medium         113          1038         107             53
Actual: High           179            95        2246            559
Actual: Extreme        321           220         743           1661


In [ ]:
# saving the model
import joblib
joblib.dump(model, "stress_model.pkl")
joblib.dump(vectorizer, "tfidf_vectorizer.pkl")

print("Model saved successfully!")

Model saved successfully!
